# SmolVLM-256M · Dual-crop baseline

| | |
|---|---|
| **Project** | EdgeVLM-Drive — resource-aware vision-language driver monitoring on NVIDIA Jetson Nano |
| **Model** | SmolVLM-256M-Instruct |
| **Stage** | 8 of 12 · Dual-crop baseline |
| **Purpose** | Paired full-scene and face-crop input with LoRA on the vision encoder and the text decoder. |
| **Input** | train / validation / test CSVs with `frame_path` and `label` (see `data/README.md`) |
| **Output** | LoRA adapter (`best_by_macro_f1`), predictions CSV, metrics |
| **Configuration** | [`08_dual_crop_baseline.json`](../../configs/smolvlm256m/08_dual_crop_baseline.json) |
| **Results** | [`dual_crop_baseline.json`](../../results/smolvlm_256m/dual_crop_baseline.json) |
| **Type** | Experiment; cell outputs from the original run are retained |
| **Setup** | set the paths in the first configuration cell to your data and split folders |


# EdgeVLM-Drive — SmolVLM-256M Phase 2 with all selected techniques

This is a new, isolated experiment. It does not overwrite the earlier 256M model.

Applied techniques:

1. Phase 2 subject-disjoint train/validation/test splits with training-path deduplication.
2. Production-aligned preprocessing imported directly from `langgraph_pipeline.py`.
3. Dual-crop input: full available frame plus face crop from the same moment.
4. LoRA on both vision-encoder and text-decoder attention projections.
5. WeightedRandomSampler for class-balanced training exposure.
6. Up to eight epochs with early stopping selected by generated validation Macro-F1.
7. Checkpoints, resume support, terminal-friendly ETA, and resumable full-test evaluation.

The default starts from the previous good 256M adapter, merges it into the base, and
then trains a fresh vision+text LoRA adapter. For a controlled comparison against the
500M experiment, set `START_FROM_PREVIOUS_ADAPTER=False` so both models start from
their untouched pretrained bases.


## 1. Configuration and production preprocessing


In [1]:
import os
import sys
import importlib.util
from pathlib import Path

# Keep this experiment on one GPU. This avoids the DataParallel stall observed earlier.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HOME = Path("/home/jovyan")
SPLIT_DIR = HOME / "phase2-tain-val-test-splits"
TRAIN_CSV = SPLIT_DIR / "final_train.csv"
VAL_CSV = SPLIT_DIR / "final_val.csv"
TEST_CSV = SPLIT_DIR / "final_test.csv"

PROJECT_ROOT = HOME / "EdgeVLM_finetune vlm256"
OUTPUT_ROOT = PROJECT_ROOT / "phase2_256m_all_techniques"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
LAST_ADAPTER_DIR = OUTPUT_ROOT / "last_adapter"
BEST_ADAPTER_DIR = OUTPUT_ROOT / "best_by_macro_f1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
MODEL_CACHE_CANDIDATES = [
    PROJECT_ROOT / "model_cache",
    HOME / "EdgeVLM_finetune" / "model_cache",
]
MODEL_CACHE = next((p for p in MODEL_CACHE_CANDIDATES if p.exists()), MODEL_CACHE_CANDIDATES[0])
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

PREVIOUS_ADAPTER_CANDIDATES = [
    PROJECT_ROOT / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune_vlm256" / "smolvlm_256m_lora_augmented_classweighted",
    HOME / "EdgeVLM_finetune" / "smolvlm_256m_lora_augmented_classweighted",
]
PREVIOUS_ADAPTER_DIR = next(
    (p for p in PREVIOUS_ADAPTER_CANDIDATES if (p / "adapter_model.safetensors").is_file()),
    None,
)

START_FROM_PREVIOUS_ADAPTER = True
LABELS = ["NORMAL", "DROWSY", "LOOKING_AWAY", "PHONE_USAGE"]
IMAGE_SIZE = 512
MAX_EPOCHS = 8
EARLY_STOPPING_PATIENCE = 2
MACRO_F1_SAMPLES_PER_CLASS = 80
SEED = 42

LANGGRAPH_MODULE_PATH = HOME / "EdgeVLM_finetune" / "langgraph_pipeline.py"
if not LANGGRAPH_MODULE_PATH.is_file():
    raise FileNotFoundError(
        f"Production preprocessing module not found: {LANGGRAPH_MODULE_PATH}"
    )

spec = importlib.util.spec_from_file_location("langgraph_pipeline", LANGGRAPH_MODULE_PATH)
lgp = importlib.util.module_from_spec(spec)
sys.modules["langgraph_pipeline"] = lgp
spec.loader.exec_module(lgp)

print("LangGraph preprocessing:", lgp.__file__)
print("Full-frame size:", lgp.FULL_FRAME_SIZE)
print("Face-crop size:", lgp.FACE_CROP_SIZE)
print("Haar fallback enabled:", lgp.ENABLE_HAAR_FACE_FALLBACK)
print("Previous adapter:", PREVIOUS_ADAPTER_DIR)
print("Use previous adapter:", START_FROM_PREVIOUS_ADAPTER)
print("Output directory:", OUTPUT_ROOT)
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(path, "exists:", path.is_file())
    if not path.is_file():
        raise FileNotFoundError(path)


LangGraph preprocessing: /home/jovyan/EdgeVLM_finetune/langgraph_pipeline.py
Full-frame size: (448, 448)
Face-crop size: (256, 256)
Haar fallback enabled: True
Previous adapter: /home/jovyan/EdgeVLM_finetune vlm256/smolvlm_256m_lora_augmented_classweighted
Use previous adapter: True
Output directory: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_all_techniques
/home/jovyan/phase2-tain-val-test-splits/final_train.csv exists: True
/home/jovyan/phase2-tain-val-test-splits/final_val.csv exists: True
/home/jovyan/phase2-tain-val-test-splits/final_test.csv exists: True


## 2. Read, validate, and deduplicate Phase 2 data


In [2]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)
datasets = {"train": train_df, "validation": val_df, "test": test_df}
required_columns = {"frame_path", "label", "source_dataset", "subject"}


def resolve_image_path(value):
    path = Path(str(value).strip().replace("\\", "/"))
    return path if path.is_absolute() else HOME / path


for split_name, dataframe in datasets.items():
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{split_name} missing columns: {sorted(missing_columns)}")
    dataframe["label"] = dataframe["label"].astype(str).str.strip().str.upper()
    unknown = sorted(set(dataframe["label"]) - set(LABELS))
    if unknown:
        raise ValueError(f"{split_name} contains unknown labels: {unknown}")
    dataframe["resolved_path"] = dataframe["frame_path"].map(
        lambda value: str(resolve_image_path(value))
    )
    missing = [p for p in dataframe["resolved_path"] if not Path(p).is_file()]
    print(f"\n{split_name}: {len(dataframe)} rows | missing images: {len(missing)}")
    print(dataframe["label"].value_counts().reindex(LABELS, fill_value=0))
    if missing:
        print("First missing path:", missing[0])

if any(not Path(p).is_file() for df in datasets.values() for p in df["resolved_path"]):
    raise FileNotFoundError("Fix missing image paths before training.")

for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    image_overlap = set(datasets[left]["resolved_path"]) & set(datasets[right]["resolved_path"])
    print(f"Exact image overlap {left}/{right}:", len(image_overlap))
    if image_overlap:
        raise ValueError(f"Image leakage detected between {left} and {right}")


def subject_keys(dataframe):
    return set(zip(dataframe["source_dataset"].astype(str), dataframe["subject"].astype(str)))


for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    overlap = subject_keys(datasets[left]) & subject_keys(datasets[right])
    print(f"Subject overlap {left}/{right}:", len(overlap))
    if overlap:
        raise ValueError(f"Subject leakage detected between {left} and {right}: {list(overlap)[:5]}")

before = len(train_df)
train_df = train_df.drop_duplicates(subset=["resolved_path"], keep="first").reset_index(drop=True)
datasets["train"] = train_df
print("\nTraining duplicates removed:", before - len(train_df))
print("Final training rows:", len(train_df))
print(train_df["label"].value_counts().reindex(LABELS, fill_value=0))



train: 27872 rows | missing images: 0
label
NORMAL          13992
DROWSY           6287
LOOKING_AWAY     3028
PHONE_USAGE      4565
Name: count, dtype: int64

validation: 5761 rows | missing images: 0
label
NORMAL          3418
DROWSY          1161
LOOKING_AWAY     310
PHONE_USAGE      872
Name: count, dtype: int64



test: 5558 rows | missing images: 0
label
NORMAL          3277
DROWSY          1021
LOOKING_AWAY     302
PHONE_USAGE      958
Name: count, dtype: int64


Exact image overlap train/validation: 0
Exact image overlap train/test: 0
Exact image overlap validation/test: 0
Subject overlap train/validation: 0
Subject overlap train/test: 0
Subject overlap validation/test: 0

Training duplicates removed: 2022
Final training rows: 25850
label
NORMAL          13992
DROWSY           5060
LOOKING_AWAY     2233
PHONE_USAGE      4565
Name: count, dtype: int64


## 3. Verify dual-crop preprocessing on real images


In [3]:
import cv2
import numpy as np
from PIL import Image


def create_dual_crops(image_path):
    raw_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if raw_bgr is None:
        raise FileNotFoundError(f"OpenCV could not read: {image_path}")
    enhanced_rgb, frame_quality_ok = lgp.enhance_frame(raw_bgr)
    full_crop = lgp.resize_full_frame(enhanced_rgb)
    face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)
    face_detected = lgp.detect_face_bbox_haar(enhanced_rgb) is not None
    return full_crop, face_crop, bool(frame_quality_ok), face_detected


RUN_PREPROCESSING_SANITY_CHECK = True
if RUN_PREPROCESSING_SANITY_CHECK:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    visual_rows = pd.concat([
        train_df[train_df["label"] == label].sample(n=2, random_state=SEED)
        for label in LABELS
    ]).reset_index(drop=True)

    fig, axes = plt.subplots(len(visual_rows), 2, figsize=(7, 3 * len(visual_rows)))
    for index, row in visual_rows.iterrows():
        full_crop, face_crop, quality_ok, face_detected = create_dual_crops(row["resolved_path"])
        axes[index, 0].imshow(full_crop)
        axes[index, 0].set_title(f"{row['label']} | full available frame")
        axes[index, 1].imshow(face_crop)
        axes[index, 1].set_title(f"face crop | detected={face_detected}")
        axes[index, 0].axis("off")
        axes[index, 1].axis("off")
    plt.tight_layout()
    sanity_path = OUTPUT_ROOT / "preprocessing_sanity_check.png"
    fig.savefig(sanity_path, dpi=110)
    plt.close(fig)
    print("Saved visual check:", sanity_path)

    print("\nHaar face-detection rate on 100 training images per class:")
    for label in LABELS:
        subset = train_df[train_df["label"] == label]
        sample = subset.sample(n=min(100, len(subset)), random_state=SEED)
        detected = 0
        for path in sample["resolved_path"]:
            raw_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
            enhanced_rgb, _ = lgp.enhance_frame(raw_bgr)
            detected += int(lgp.detect_face_bbox_haar(enhanced_rgb) is not None)
        print(f"{label}: {100 * detected / len(sample):.1f}%")
else:
    print("Preprocessing sanity check skipped.")


Saved visual check: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_all_techniques/preprocessing_sanity_check.png

Haar face-detection rate on 100 training images per class:


NORMAL: 98.0%


DROWSY: 94.0%


LOOKING_AWAY: 86.0%


PHONE_USAGE: 95.0%


The `full_crop` is the complete **available input file**. If the dataset file is
already a face-only frame, this step cannot recreate missing hands or cabin context.
The saved sanity-check image must therefore be reviewed before trusting PHONE_USAGE
or LOOKING_AWAY performance.


## 4. Verify the existing offline training environment


In [4]:
import torch
import transformers
import tokenizers
import peft
import accelerate
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel, LoraConfig, TaskType, get_peft_model

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook.")
print("Visible GPUs:", torch.cuda.device_count())
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("OpenCV:", cv2.__version__)

if transformers.__version__ != "4.46.3":
    raise RuntimeError("Use Transformers 4.46.3 for this reproducible experiment.")
if tokenizers.__version__ != "0.20.3":
    raise RuntimeError("Use Tokenizers 0.20.3 for this reproducible experiment.")


Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
PyTorch: 2.11.0+cu126
CUDA available: True
Visible GPUs: 1
GPU: NVIDIA A16
Transformers: 4.46.3
Tokenizers: 0.20.3
PEFT: 0.20.0
Accelerate: 1.15.0
OpenCV: 4.10.0


## 5. Build a 256M starting model and attach vision+text LoRA


In [5]:
import gc

device = torch.device("cuda:0")
dtype = torch.float16


def load_plain_base():
    options = {
        "cache_dir": str(MODEL_CACHE),
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }
    try:
        return AutoModelForVision2Seq.from_pretrained(
            MODEL_ID, local_files_only=True, **options
        )
    except OSError:
        print("Local cache incomplete; attempting to download missing base-model files...")
        return AutoModelForVision2Seq.from_pretrained(MODEL_ID, **options)


def load_starting_base():
    base = load_plain_base()
    if START_FROM_PREVIOUS_ADAPTER:
        if PREVIOUS_ADAPTER_DIR is None:
            raise FileNotFoundError(
                "START_FROM_PREVIOUS_ADAPTER=True but the previous adapter was not found."
            )
        previous_model = PeftModel.from_pretrained(base, str(PREVIOUS_ADAPTER_DIR))
        base = previous_model.merge_and_unload()
        print("Merged previous good text-LoRA adapter into the 256M starting base.")
    else:
        print("Starting from the untouched pretrained 256M base.")
    return base


processor_source = (
    str(PREVIOUS_ADAPTER_DIR)
    if START_FROM_PREVIOUS_ADAPTER and PREVIOUS_ADAPTER_DIR is not None
    else MODEL_ID
)
processor = AutoProcessor.from_pretrained(
    processor_source,
    cache_dir=str(MODEL_CACHE),
    local_files_only=True,
)
model = load_starting_base()

ATTENTION_SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj", "out_proj")
text_targets, vision_targets = [], []
for name, module in model.named_modules():
    if not isinstance(module, torch.nn.Linear):
        continue
    if not name.endswith(ATTENTION_SUFFIXES):
        continue
    if ".text_model." in name:
        text_targets.append(name)
    elif ".vision_model." in name:
        vision_targets.append(name)

print("Text attention targets:", len(text_targets))
print("Vision attention targets:", len(vision_targets))
if not text_targets or not vision_targets:
    raise RuntimeError(
        "Could not find both text and vision attention projections. "
        "Inspect model.named_modules() before training."
    )

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=sorted(text_targets + vision_targets),
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, lora_config).to(device)
model.train()
model.print_trainable_parameters()


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Merged previous good text-LoRA adapter into the 256M starting base.
Text attention targets: 120
Vision attention targets: 48


trainable params: 3,022,848 || all params: 259,507,776 || trainable%: 1.1648


## 6. Dual-crop dataset and multimodal collator


In [6]:
from torch.utils.data import Dataset, DataLoader

INSTRUCTION = (
    "You are shown two images of the same moment from an in-cabin driver camera. "
    "The first image is the full available frame. The second image is a close-up "
    "of the driver's face for eye and head-pose detail. Using both images together, "
    "return exactly one label from: NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
    "Do not provide an explanation or punctuation."
)


class DriverDualCropDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        full_crop, face_crop, quality_ok, face_detected = create_dual_crops(row["resolved_path"])
        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "image"},
                {"type": "text", "text": INSTRUCTION},
            ],
        }]
        return {
            "full_image": Image.fromarray(full_crop).convert("RGB"),
            "face_image": Image.fromarray(face_crop).convert("RGB"),
            "messages": messages,
            "target": row["label"],
            "path": row["resolved_path"],
        }


class DualCropCollator:
    def __init__(self, processor, image_size):
        self.processor = processor
        self.image_size = image_size

    def __call__(self, features):
        images = [[item["full_image"], item["face_image"]] for item in features]
        prompt_texts, full_texts = [], []
        for item in features:
            prompt_messages = item["messages"]
            full_messages = prompt_messages + [{
                "role": "assistant",
                "content": [{"type": "text", "text": item["target"]}],
            }]
            prompt_texts.append(self.processor.apply_chat_template(
                prompt_messages, add_generation_prompt=True, tokenize=False
            ))
            full_texts.append(self.processor.apply_chat_template(
                full_messages, add_generation_prompt=False, tokenize=False
            ))

        kwargs = {
            "return_tensors": "pt",
            "padding": True,
            "truncation": True,
            "size": {"longest_edge": self.image_size},
            "max_image_size": {"longest_edge": self.image_size},
        }
        batch = self.processor(text=full_texts, images=images, **kwargs)
        prompt_batch = self.processor(text=prompt_texts, images=images, **kwargs)
        labels = batch["input_ids"].clone()
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)
        for row_index, prompt_length in enumerate(prompt_lengths):
            labels[row_index, : int(prompt_length)] = -100
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100
        batch["labels"] = labels
        return batch


train_dataset = DriverDualCropDataset(train_df)
val_dataset = DriverDualCropDataset(val_df)
test_dataset = DriverDualCropDataset(test_df)
collator = DualCropCollator(processor, IMAGE_SIZE)

sample_batch = collator([train_dataset[0]])
print("pixel_values:", tuple(sample_batch["pixel_values"].shape))
if sample_batch["pixel_values"].shape[1] != 2:
    raise RuntimeError("Dual-crop validation failed: expected two images per sample.")

# Backward-pass smoke test without an optimizer step, so it does not modify weights.
sample_batch = {
    key: value.to(device) if torch.is_tensor(value) else value
    for key, value in sample_batch.items()
}
smoke_output = model(**sample_batch)
smoke_output.loss.backward()
model.zero_grad(set_to_none=True)
print("Non-updating dual-crop smoke-test loss:", float(smoke_output.loss.detach().cpu()))
del sample_batch, smoke_output
gc.collect()
torch.cuda.empty_cache()


pixel_values: (1, 2, 3, 512, 512)


Non-updating dual-crop smoke-test loss: 0.5296772718429565


## 7. Generated Macro-F1 model selection and early stopping


In [7]:
import json
import time
from sklearn.metrics import f1_score
from transformers import TrainerCallback


def parse_prediction(raw_text):
    normalized = str(raw_text).strip().upper()
    exact = normalized in LABELS
    cleaned = normalized.strip(" .,;:!?'\"")
    return (cleaned if cleaned in LABELS else "INVALID"), exact


def predict_rows_batched(model_ref, processor_ref, rows_df, batch_size=8):
    predictions = []
    old_padding = processor_ref.tokenizer.padding_side
    processor_ref.tokenizer.padding_side = "left"
    try:
        for start in range(0, len(rows_df), batch_size):
            rows = rows_df.iloc[start : start + batch_size]
            images, prompts = [], []
            for _, row in rows.iterrows():
                full_crop, face_crop, _, _ = create_dual_crops(row["resolved_path"])
                images.append([
                    Image.fromarray(full_crop).convert("RGB"),
                    Image.fromarray(face_crop).convert("RGB"),
                ])
                messages = [{
                    "role": "user",
                    "content": [
                        {"type": "image"}, {"type": "image"},
                        {"type": "text", "text": INSTRUCTION},
                    ],
                }]
                prompts.append(processor_ref.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False
                ))

            inputs = processor_ref(
                text=prompts,
                images=images,
                return_tensors="pt",
                padding=True,
                size={"longest_edge": IMAGE_SIZE},
                max_image_size={"longest_edge": IMAGE_SIZE},
            )
            inputs = {
                key: value.to(next(model_ref.parameters()).device)
                if torch.is_tensor(value) else value
                for key, value in inputs.items()
            }
            with torch.inference_mode():
                generated = model_ref.generate(
                    **inputs, max_new_tokens=12, do_sample=False
                )
            input_length = inputs["input_ids"].shape[-1]
            raw_outputs = processor_ref.batch_decode(
                generated[:, input_length:], skip_special_tokens=True
            )
            predictions.extend(parse_prediction(text)[0] for text in raw_outputs)
    finally:
        processor_ref.tokenizer.padding_side = old_padding
    return predictions


macro_rows = []
for label in LABELS:
    class_rows = val_df[val_df["label"] == label]
    count = min(MACRO_F1_SAMPLES_PER_CLASS, len(class_rows))
    macro_rows.append(class_rows.sample(n=count, random_state=SEED))
macro_f1_df = pd.concat(macro_rows).reset_index(drop=True)
print("Macro-F1 validation subset:", macro_f1_df["label"].value_counts().to_dict())


class MacroF1EarlyStopping(TrainerCallback):
    def __init__(self, processor_ref, check_df, best_dir, patience):
        self.processor_ref = processor_ref
        self.check_df = check_df
        self.best_dir = Path(best_dir)
        self.patience = int(patience)
        self.log_path = OUTPUT_ROOT / "macro_f1_epoch_log.json"
        self.history = []
        self.best_macro_f1 = -1.0
        self.epochs_without_improvement = 0
        if self.log_path.is_file():
            self.history = json.loads(self.log_path.read_text(encoding="utf-8"))
            if self.history:
                scores = [float(row["macro_f1"]) for row in self.history]
                self.best_macro_f1 = max(scores)
                best_index = max(range(len(scores)), key=scores.__getitem__)
                self.epochs_without_improvement = len(scores) - 1 - best_index

    def on_epoch_end(self, args, state, control, **kwargs):
        model_ref = kwargs.get("model")
        if model_ref is None:
            return control
        model_ref.eval()
        started = time.perf_counter()
        try:
            predicted = predict_rows_batched(
                model_ref, self.processor_ref, self.check_df, batch_size=8
            )
            true = self.check_df["label"].astype(str).tolist()
            macro_f1 = f1_score(
                true, predicted, labels=LABELS, average="macro", zero_division=0
            )
            per_class = f1_score(
                true, predicted, labels=LABELS, average=None, zero_division=0
            )
            improved = macro_f1 > self.best_macro_f1
            if improved:
                self.best_macro_f1 = float(macro_f1)
                self.epochs_without_improvement = 0
                self.best_dir.mkdir(parents=True, exist_ok=True)
                model_ref.save_pretrained(str(self.best_dir))
                self.processor_ref.save_pretrained(str(self.best_dir))
            else:
                self.epochs_without_improvement += 1

            record = {
                "epoch": float(state.epoch),
                "macro_f1": float(macro_f1),
                "per_class_f1": dict(zip(LABELS, map(float, per_class))),
                "improved": bool(improved),
                "seconds": float(time.perf_counter() - started),
            }
            self.history.append(record)
            self.log_path.write_text(json.dumps(self.history, indent=2), encoding="utf-8")
            print(
                f"Macro-F1 epoch {state.epoch:.2f}: {macro_f1:.4f} | "
                f"best {self.best_macro_f1:.4f} | per class "
                f"{dict(zip(LABELS, np.round(per_class, 3)))}",
                flush=True,
            )
            if self.epochs_without_improvement >= self.patience:
                print("Macro-F1 patience exhausted; stopping training.", flush=True)
                control.should_training_stop = True
        finally:
            model_ref.train()
        return control


Macro-F1 validation subset: {'NORMAL': 80, 'DROWSY': 80, 'LOOKING_AWAY': 80, 'PHONE_USAGE': 80}


## 8. Terminal-friendly training ETA


In [8]:
from datetime import datetime, timedelta

PROGRESS_FILE = OUTPUT_ROOT / "training_progress.txt"
PROGRESS_HISTORY = OUTPUT_ROOT / "training_progress_history.log"


def format_duration(seconds):
    hours, remainder = divmod(max(0, int(seconds)), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


class ETAProgressCallback(TrainerCallback):
    def __init__(self, update_every=25):
        self.update_every = int(update_every)
        self.start_time = None
        self.start_step = 0

    def _write(self, message):
        PROGRESS_FILE.write_text(message + "\n", encoding="utf-8")
        with PROGRESS_HISTORY.open("a", encoding="utf-8") as handle:
            handle.write(message + "\n")
        print(message, flush=True)

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        self.start_step = int(state.global_step)
        self._write(
            f"Training started | step {state.global_step}/{state.max_steps} | "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        )

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step)
        if step != int(state.max_steps) and step % self.update_every:
            return
        elapsed = time.time() - self.start_time
        completed = max(1, step - self.start_step)
        seconds_per_step = elapsed / completed
        remaining = max(0, int(state.max_steps) - step) * seconds_per_step
        percentage = 100 * step / max(1, int(state.max_steps))
        finish = datetime.now() + timedelta(seconds=remaining)
        self._write(
            f"{percentage:6.2f}% | step {step}/{state.max_steps} | "
            f"elapsed {format_duration(elapsed)} | remaining upper-bound "
            f"{format_duration(remaining)} | ETA {finish.strftime('%Y-%m-%d %H:%M:%S')}"
        )


## 9. Balanced multi-epoch fine-tuning


In [9]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler
from transformers import TrainingArguments, Trainer, set_seed
from transformers.trainer_utils import get_last_checkpoint

set_seed(SEED)


class BalancedTrainer(Trainer):
    def _get_train_sampler(self):
        labels = self.train_dataset.dataframe["label"].astype(str).tolist()
        counts = Counter(labels)
        weights = torch.tensor(
            [1.0 / counts[label] for label in labels], dtype=torch.double
        )
        return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=MAX_EPOCHS,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    disable_tqdm=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
    report_to="none",
    seed=SEED,
)

macro_callback = MacroF1EarlyStopping(
    processor_ref=processor,
    check_df=macro_f1_df,
    best_dir=BEST_ADAPTER_DIR,
    patience=EARLY_STOPPING_PATIENCE,
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=[ETAProgressCallback(update_every=25), macro_callback],
)

last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
print("Resume checkpoint:", last_checkpoint)
print("Starting balanced dual-crop vision+text LoRA training...")
print("Progress file:", PROGRESS_FILE)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
print("Training ended:", train_result)
trainer.save_model(str(LAST_ADAPTER_DIR))
processor.save_pretrained(str(LAST_ADAPTER_DIR))
print("Last adapter:", LAST_ADAPTER_DIR)
print("Macro-F1-best adapter:", BEST_ADAPTER_DIR)


Resume checkpoint: None
Starting balanced dual-crop vision+text LoRA training...
Progress file: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_all_techniques/training_progress.txt
Training started | step 0/25848 | 2026-09-26 23:42:50


  0.10% | step 25/25848 | elapsed 00:02:42 | remaining upper-bound 46:42:25 | ETA 2026-09-28 22:27:58


  0.19% | step 50/25848 | elapsed 00:05:24 | remaining upper-bound 46:29:35 | ETA 2026-09-28 22:17:50


{'loss': 0.2767, 'grad_norm': 42.885074615478516, 'learning_rate': 3.5576179427687554e-06, 'epoch': 0.015473887814313346}


  0.29% | step 75/25848 | elapsed 00:08:05 | remaining upper-bound 46:21:44 | ETA 2026-09-28 22:12:40


...

Macro-F1 epoch 3.00: 0.7790 | best 0.8142 | per class {'NORMAL': np.float64(0.727), 'DROWSY': np.float64(0.808), 'LOOKING_AWAY': np.float64(0.793), 'PHONE_USAGE': np.float64(0.788)}


Macro-F1 patience exhausted; stopping training.


{'eval_loss': 0.24237309396266937, 'eval_runtime': 2010.0275, 'eval_samples_per_second': 2.866, 'eval_steps_per_second': 2.866, 'epoch': 2.9997678916827852}


{'train_runtime': 67959.9152, 'train_samples_per_second': 3.043, 'train_steps_per_second': 0.38, 'train_loss': 0.04914369564559159, 'epoch': 2.9997678916827852}
Training ended: TrainOutput(global_step=9693, training_loss=0.04914369564559159, metrics={'train_runtime': 67959.9152, 'train_samples_per_second': 3.043, 'train_steps_per_second': 0.38, 'train_loss': 0.04914369564559159, 'epoch': 2.9997678916827852})


Last adapter: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_all_techniques/last_adapter
Macro-F1-best adapter: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_all_techniques/best_by_macro_f1


## 10. Reload the Macro-F1-best model


In [10]:
if not (BEST_ADAPTER_DIR / "adapter_model.safetensors").is_file():
    raise FileNotFoundError("Macro-F1-best adapter was not saved.")

del trainer, model
gc.collect()
torch.cuda.empty_cache()

eval_processor = AutoProcessor.from_pretrained(str(BEST_ADAPTER_DIR), local_files_only=True)
eval_base = load_starting_base()
eval_model = PeftModel.from_pretrained(eval_base, str(BEST_ADAPTER_DIR)).to(device)
eval_model.eval()
eval_model.config.use_cache = True
print("Loaded Macro-F1-best model:", BEST_ADAPTER_DIR)


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Merged previous good text-LoRA adapter into the 256M starting base.


Loaded Macro-F1-best model: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_all_techniques/best_by_macro_f1


## 11. Full, resumable Phase 2 test evaluation


In [11]:
from tqdm.auto import tqdm

PREDICTIONS_CSV = OUTPUT_ROOT / "phase2_256m_all_techniques_test_predictions.csv"
SAVE_EVERY = 250


def predict_test_row(row):
    full_crop, face_crop, _, _ = create_dual_crops(row["resolved_path"])
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"}, {"type": "image"},
            {"type": "text", "text": INSTRUCTION},
        ],
    }]
    prompt = eval_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = eval_processor(
        text=prompt,
        images=[[Image.fromarray(full_crop), Image.fromarray(face_crop)]],
        return_tensors="pt",
        size={"longest_edge": IMAGE_SIZE},
        max_image_size={"longest_edge": IMAGE_SIZE},
    )
    inputs = {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }
    started = time.perf_counter()
    with torch.inference_mode():
        generated = eval_model.generate(
            **inputs, max_new_tokens=12, do_sample=False
        )
    latency_ms = (time.perf_counter() - started) * 1000
    input_length = inputs["input_ids"].shape[-1]
    raw = eval_processor.batch_decode(
        generated[:, input_length:], skip_special_tokens=True
    )[0].strip()
    prediction, exact = parse_prediction(raw)
    return raw, prediction, exact, latency_ms


if PREDICTIONS_CSV.is_file():
    records = pd.read_csv(PREDICTIONS_CSV).to_dict("records")
    finished = {int(row["row_index"]) for row in records}
    print("Resuming after", len(finished), "test rows.")
else:
    records, finished = [], set()

torch.cuda.reset_peak_memory_stats()
remaining = [index for index in range(len(test_df)) if index not in finished]
for completed, index in enumerate(
    tqdm(remaining, desc="Evaluating full Phase 2 test set", mininterval=2), start=1
):
    row = test_df.iloc[index]
    raw, prediction, exact, latency_ms = predict_test_row(row)
    records.append({
        "row_index": index,
        "frame_path": row["frame_path"],
        "true_label": row["label"],
        "raw_output": raw,
        "prediction": prediction,
        "exact_label_valid": exact,
        "latency_ms": latency_ms,
    })
    if completed % SAVE_EVERY == 0:
        pd.DataFrame(records).sort_values("row_index").to_csv(PREDICTIONS_CSV, index=False)

predictions_df = (
    pd.DataFrame(records)
    .sort_values("row_index")
    .drop_duplicates("row_index", keep="last")
)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
print("Saved test rows:", len(predictions_df), "/", len(test_df))
print("Predictions:", PREDICTIONS_CSV)


Evaluating full Phase 2 test set:   0%|          | 0/5558 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Saved test rows: 5558 / 5558
Predictions: /home/jovyan/EdgeVLM_finetune vlm256/phase2_256m_all_techniques/phase2_256m_all_techniques_test_predictions.csv


## 12. Final metrics


In [12]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

if len(predictions_df) != len(test_df):
    raise RuntimeError("Full test evaluation is incomplete.")

y_true = predictions_df["true_label"].astype(str).to_numpy()
y_pred = predictions_df["prediction"].astype(str).to_numpy()
latencies = predictions_df["latency_ms"].astype(float).to_numpy()

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=LABELS, zero_division=0
)
unsafe_true = np.isin(y_true, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])
normal_true = y_true == "NORMAL"
unsafe_pred = np.isin(y_pred, ["DROWSY", "LOOKING_AWAY", "PHONE_USAGE"])

unsafe_to_safe_miss = np.mean(np.isin(y_pred[unsafe_true], ["NORMAL", "INVALID"]))
unsafe_class_error = np.mean(y_pred[unsafe_true] != y_true[unsafe_true])
false_alarm = np.mean(unsafe_pred[normal_true])
exact_validity = predictions_df["exact_label_valid"].astype(bool).mean()
parser_failure = np.mean(y_pred == "INVALID")
matrix_labels = LABELS + ["INVALID"]
cm = confusion_matrix(y_true, y_pred, labels=matrix_labels)

print("\n========== 256M ALL-TECHNIQUES PHASE 2 TEST ==========")
print("Images:", len(predictions_df))
print("Accuracy:", round(accuracy_score(y_true, y_pred) * 100, 2), "%")
print("Balanced accuracy:", round(recall.mean() * 100, 2), "%")
print("Macro-F1:", round(f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0) * 100, 2), "%")
print("Exact-label validity:", round(exact_validity * 100, 2), "%")
print("Parser failure:", round(parser_failure * 100, 2), "%")
print("\nPer-class results:")
for label, p, r, score, count in zip(LABELS, precision, recall, f1, support):
    print(f"{label}: precision={p:.3f}, recall={r:.3f}, F1={score:.3f}, samples={count}")
print("\nConfusion matrix:")
print(pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels))
print("Unsafe-to-safe miss rate:", round(unsafe_to_safe_miss * 100, 2), "%")
print("Unsafe-class error rate:", round(unsafe_class_error * 100, 2), "%")
print("False-alarm rate:", round(false_alarm * 100, 2), "%")
print("Mean latency (ms):", round(float(np.mean(latencies)), 2))
print("P95 latency (ms):", round(float(np.percentile(latencies, 95)), 2))
print("Peak GPU memory (MB):", round(torch.cuda.max_memory_allocated() / 1024**2, 2))



========== 256M ALL-TECHNIQUES PHASE 2 TEST ==========
Images: 5558
Accuracy: 83.68 %
Balanced accuracy: 82.8 %
Macro-F1: 78.16 %
Exact-label validity: 100.0 %
Parser failure: 0.0 %

Per-class results:
NORMAL: precision=0.917, recall=0.854, F1=0.885, samples=3277
DROWSY: precision=0.702, recall=0.871, F1=0.777, samples=1021
LOOKING_AWAY: precision=0.486, recall=0.851, F1=0.619, samples=302
PHONE_USAGE: precision=0.996, recall=0.736, F1=0.846, samples=958

Confusion matrix:
              NORMAL  DROWSY  LOOKING_AWAY  PHONE_USAGE  INVALID
NORMAL          2800     332           143            2        0
DROWSY           123     889             8            1        0
LOOKING_AWAY      39       6           257            0        0
PHONE_USAGE       92      40           121          705        0
INVALID            0       0             0            0        0
Unsafe-to-safe miss rate: 11.14 %
Unsafe-class error rate: 18.85 %
False-alarm rate: 14.56 %
Mean latency (ms): 299.03
P95 latency 

## 13. Optional merged FP32 deployment export


In [13]:
CREATE_MERGED_FP32 = False

if CREATE_MERGED_FP32:
    MERGED_DIR = OUTPUT_ROOT / "merged_model_256m_all_techniques_fp32"
    merged_model = eval_model.merge_and_unload().float()
    merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    eval_processor.save_pretrained(str(MERGED_DIR))
    print("Merged FP32 model:", MERGED_DIR)
else:
    print("Merge skipped until the new test results are accepted.")


Merge skipped until the new test results are accepted.
